# Pilot data collection

Collects a first batch of Play Store reviews across four e-commerce apps, balanced by star rating, and checks how many are code-mixed Hinglish.

## Setup
Installs `google-play-scraper`.

In [ ]:
!pip install google-play-scraper -q

## App list
App IDs resolved from their Play Store URLs (the string after `id=`).

In [ ]:
from google_play_scraper import app, reviews, Sort
import pandas as pd
import time
import re

APPS = {
    "Meesho":  "com.meesho.supply",
    "Nykaa":   "com.fsn.nykaa",
    "Blinkit": "com.grofers.customerapp",
    "Zepto":   "com.zeptoconsumerapp",
}

for name, app_id in APPS.items():
    try:
        info = app(app_id, lang="en", country="in")
        print(f"OK    {name}: {info['title']}")
    except Exception as e:
        print(f"FAIL  {name} ({app_id}) -> please check this ID")

## Scraping function
Pulls reviews per app, per star rating (1-5), to avoid the natural skew toward 5-star reviews. Queries both the English and Hindi store-language settings, since Hinglish reviews surface under either.

In [ ]:
PER_STAR = 150

def scrape_app(name, app_id):
    rows = []
    for lang in ["en", "hi"]:
        for star in [1, 2, 3, 4, 5]:
            try:
                result, _ = reviews(
                    app_id,
                    lang=lang,
                    country="in",
                    sort=Sort.NEWEST,
                    count=PER_STAR,
                    filter_score_with=star,
                )
            except Exception as e:
                print(f"  error: {name} | lang={lang} | {star} star: {e}")
                continue
            for r in result:
                rows.append({
                    "app": name,
                    "review_id": r["reviewId"],
                    "text": r["content"],
                    "stars": r["score"],
                    "date": r["at"],
                    "lang_setting": lang,
                })
            print(f"  {name} | lang={lang} | {star} star: {len(result)} reviews")
            time.sleep(1)   # small pause so we don't overload the server
    return rows

## Collection and cleaning
De-duplicates, drops empty reviews, and drops reviews under four words (too short to mention a specific aspect).

In [ ]:
all_rows = []
for name, app_id in APPS.items():
    print("Scraping", name)
    all_rows += scrape_app(name, app_id)

df = pd.DataFrame(all_rows)
print("\nRaw reviews collected:", len(df))

df = df.drop_duplicates(subset="review_id")
df = df[df["text"].notna()]
df["n_words"] = df["text"].str.split().str.len()
df = df[df["n_words"] >= 4]
print("After cleaning:", len(df))

## Hinglish detection
Keyword heuristic: a review is flagged `hinglish` if it contains two or more common Hindi words written in Roman script (e.g. *hai, nahi, bahut, accha*), `maybe` with exactly one, otherwise `english` or `devanagari`.

In [ ]:
HINDI_WORDS = {
    "hai", "hain", "nahi", "nhi", "nahin", "bahut", "bhut", "bohot",
    "accha", "acha", "achha", "achchha", "bekar", "bakwas", "kya", "kyu", "kyun",
    "mujhe", "mera", "meri", "mere", "aur", "toh", "tha", "thi", "raha", "rahi",
    "gaya", "gayi", "diya", "liya", "kar", "karo", "karte", "wala", "wali",
    "paisa", "paise", "sahi", "ekdam", "bilkul", "koi", "kuch", "abhi", "bhi",
    "sab", "hota", "hoti", "yaar", "bhai", "ji", "ki", "ka", "ke", "ko",
}

def classify(text):
    if re.search(r"[\u0900-\u097F]", text):
        return "devanagari"
    words = re.findall(r"[a-z]+", text.lower())
    hindi_count = sum(w in HINDI_WORDS for w in words)
    if hindi_count >= 2:
        return "hinglish"
    if hindi_count == 1:
        return "maybe"
    return "english"

df["type"] = df["text"].apply(classify)
print("Done")

## Summary statistics
Per-star and per-language-type counts, used in the dataset section of the report.

In [ ]:
print("=== Reviews per star rating ===")
print(df["stars"].value_counts().sort_index())

print("\n=== Type of language ===")
print(df["type"].value_counts())

print("\n=== Language type per app ===")
print(pd.crosstab(df["app"], df["type"]))

print("\n=== Hinglish reviews per star ===")
print(pd.crosstab(df["type"], df["stars"]))

hing = df[df["type"] == "hinglish"]
print("\n=== 10 example Hinglish reviews ===")
for t in hing.sample(min(10, len(hing)), random_state=1)["text"]:
    print("-", t)

## Export
Saves `pilot_reviews.csv`.

In [ ]:
df.to_csv("pilot_reviews.csv", index=False)
from google.colab import files
files.download("pilot_reviews.csv")